<div align='center'>

# 🎙️ Elbek Studiya — RVC Model Trainer
### Retrieval-based Voice Conversion v2 • Gradio UI + ngrok

| Qadam | Nima qiladi |
|:-----:|:------------|
| **1** | Muhit o'rnatish — `Bebra777228/TrainVocModel-EN` asosida |
| **2** | Gradio UI + ngrok ishga tushirish |
| **3** | Ovoz yuklang → Nom kiriting → Train bosing |
| **4** | **.pth + .index → ZIP** avtomatik |

> ⚡ **GPU**: `Runtime → Change runtime type → T4 GPU`
</div>

In [ ]:
#@title ⚙️ 1-QADAM — Muhit o'rnatish { display-mode: "form" }
#@markdown **Birinchi marta ~6 daqiqa ketadi. Faqat bir marta ishga tushiring.**

import os, sys, subprocess, torch
from IPython.display import clear_output
from ipywidgets import Button

REPO_BRANCH = "main"
ROOT_DIR    = "/content/drive/MyDrive/ElbekStudiya_RVC"
GITHUB_RAW  = f"https://raw.githubusercontent.com/Bebra777228/TrainVocModel-EN/refs/heads/{REPO_BRANCH}"

# ── GPU tekshirish ────────────────────────────────────────────────────────
print("GPU tekshirilmoqda...")
if not torch.cuda.is_available():
    raise Exception("\033[91m❌ GPU topilmadi!\033[0m\nRuntime → Change runtime type → T4 GPU")
gpu_name = torch.cuda.get_device_name(0)
print(f"\033[92m✅ GPU: {gpu_name}\033[0m")

# ── Google Drive ulash ────────────────────────────────────────────────────
from google.colab import drive
if not os.path.isdir("/content/drive"):
    print("\nGoogle Drive ulanmoqda...")
    drive.mount("/content/drive")
else:
    print("✅ Drive ulangan")

# /content/dataset papkasi
os.makedirs("/content/dataset", exist_ok=True)

# ── Repozitoriyani klonlash ───────────────────────────────────────────────
if not os.path.isdir(ROOT_DIR):
    print("\nRepozitoriya yuklanmoqda...")
    os.system(f"git clone https://github.com/Bebra777228/TrainVocModel-EN -b {REPO_BRANCH} {ROOT_DIR} &> /dev/null")
else:
    print(f"✅ Repo mavjud: {ROOT_DIR}")

os.chdir(ROOT_DIR)

# ── O'rnatish jarayoni ────────────────────────────────────────────────────
print("\nO'rnatish boshlanmoqda (~6 daqiqa)...")
print("[1/5] Python 3.10 o'rnatilmoqda...")
os.system("sudo apt update &> /dev/null")
os.system("sudo apt install python3.10 &> /dev/null")
os.system("sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1 &> /dev/null")
os.system("sudo update-alternatives --set python3 /usr/bin/python3.10 &> /dev/null")
os.system("curl -sS https://bootstrap.pypa.io/get-pip.py | python3 &> /dev/null")
sys.path.append("/usr/local/lib/python3.10/dist-packages")

print("[2/5] Fayllar yangilanmoqda...")
files = [
    "requirements.txt", "download_files.py",
    "configs/config.py",
    "infer/lib/rmvpe.py", "infer/lib/slicer.py",
    "infer/lib/train/utils.py",
    "infer/modules/train/train.py",
    "infer/modules/vc/infer_cli.py",
    "infer/lib/train/process_ckpt.py",
    "infer/modules/train/extract_f0.py",
    "infer/modules/train/preprocess.py",
    "infer/modules/train/extract_index.py",
    "infer/modules/train/extract_feature.py",
]
for f in files:
    os.makedirs(os.path.join(ROOT_DIR, os.path.dirname(f)), exist_ok=True)
    os.system(f"wget -q {GITHUB_RAW}/{f} -O {ROOT_DIR}/{f} &> /dev/null")

print("[3/5] Kutubxonalar o'rnatilmoqda...")
os.system("pip install --upgrade pip==24.0 &> /dev/null")
os.system("pip install --no-cache-dir -qq -r requirements.txt &> /dev/null")
os.system("pip install -q pyngrok gradio==3.50.2 &> /dev/null")
os.system("sudo apt install -y aria2 ffmpeg &> /dev/null")

print("[4/5] Modellar yuklanmoqda...")
os.system(f'python download_files.py "./assets/" "contentvec_base" &> /dev/null')

print("[5/5] Tozalash...")
os.system("rm -rf /content/sample_data")

clear_output()
print(f"\033[92m✅ GPU: {gpu_name}\033[0m")
print(f"\033[92m✅ Root: {ROOT_DIR}\033[0m")
print("\033[1;32m✅✅✅ MUHIT TAYYOR! 2-qadam ga o'ting.\033[0m")
Button(description="✔ Tayyor", button_style="success")

In [ ]:
#@title 🚀 2-QADAM — Gradio UI + ngrok ishga tushirish { display-mode: "form" }
#@markdown ### 🔑 ngrok token (ixtiyoriy)
#@markdown Bepul token: https://dashboard.ngrok.com/get-started/your-authtoken
#@markdown > Token bo'lmasa ham Gradio `.gradio.live` havolasi chiqadi
NGROK_TOKEN = '' #@param {type:"string"}

import gradio as gr
import os, re, json, shutil, zipfile, glob, pathlib
import subprocess
from random import shuffle
from subprocess import PIPE, STDOUT, Popen
from urllib.parse import urlparse

ROOT_DIR = "/content/drive/MyDrive/ElbekStudiya_RVC"
os.chdir(ROOT_DIR)
now_dir = os.getcwd()

# ── ngrok tunnel ─────────────────────────────────────────────────────────
if NGROK_TOKEN and NGROK_TOKEN.strip():
    try:
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = NGROK_TOKEN.strip()
        ngrok.kill()
        tunnel = ngrok.connect(7860, 'http')
        print(f'\033[1;32m🔗 ngrok URL: {tunnel.public_url}\033[0m')
    except Exception as e:
        print(f'⚠️  ngrok xato: {e}')

# ═══════════════════════════════════════════════════════════════════════
#  YORDAMCHI FUNKSIYALAR (original notebook logikasi)
# ═══════════════════════════════════════════════════════════════════════

def clean_name(s):
    return re.sub(r'[^a-zA-Z0-9_]', '_', (s or '').strip())

# Pretrained modellar manzillari (original kabi)
HUGG_PRET = "https://huggingface.co/Politrees/RVC_resources/resolve/main/pretrained/v2"
PARAM_ARIA = "--console-log-level=error -c -x 16 -s 16 -k 1M"

PRETRAIN_MODELS = {
    "Default": [
        ("{sr}/Default/f0D{sr}.pth", "default_D.pth"),
        ("{sr}/Default/f0G{sr}.pth", "default_G.pth"),
    ],
    "Snowie v3.1": [
        ("{sr}/Snowie/D_SnowieV3.1_{sr}.pth", "Snowie_D.pth"),
        ("{sr}/Snowie/G_SnowieV3.1_{sr}.pth", "Snowie_G.pth"),
    ],
}

def download_pretrain(sample_rate, pretrain_name, pretrain_outpath):
    """Pretrained modelni yuklab oladi."""
    os.system(f"rm -rf {pretrain_outpath}")
    os.makedirs(pretrain_outpath, exist_ok=True)
    models = PRETRAIN_MODELS.get(pretrain_name, PRETRAIN_MODELS["Default"])
    for path_tpl, out_name in models:
        path = path_tpl.format(sr=sample_rate)
        url  = f"{HUGG_PRET}/{path}"
        dest = f"{pretrain_outpath}/{out_name}"
        os.system(f"aria2c {PARAM_ARIA} {url} -d {pretrain_outpath} -o {out_name} &> /dev/null")
    G_file = f"{pretrain_outpath}/{models[1][1]}"
    D_file = f"{pretrain_outpath}/{models[0][1]}"
    return G_file, D_file

def build_filelist(model_name, sample_rate, exp_dir):
    """Trening uchun filelist.txt yaratadi."""
    f0_dir     = f"{exp_dir}/2a_f0"
    f0nsf_dir  = f"{exp_dir}/2b-f0nsf"
    gt_wavs    = f"{exp_dir}/0_gt_wavs"
    feature_dir= f"{exp_dir}/3_feature768"

    for d in [f0_dir, f0nsf_dir, gt_wavs, feature_dir]:
        if not os.path.exists(d):
            return False, f"Papka topilmadi: {d}"

    names = (
        set(n.split(".")[0] for n in os.listdir(gt_wavs))
        & set(n.split(".")[0] for n in os.listdir(feature_dir))
        & set(n.split(".")[0] for n in os.listdir(f0_dir))
        & set(n.split(".")[0] for n in os.listdir(f0nsf_dir))
    )
    if not names:
        return False, "Mos fayllar topilmadi (preprocessing to'liq bo'lmagan)"

    opt = []
    for name in names:
        opt.append(
            f"{gt_wavs}/{name}.wav|"
            f"{feature_dir}/{name}.npy|"
            f"{f0_dir}/{name}.wav.npy|"
            f"{f0nsf_dir}/{name}.wav.npy|0"
        )
    # Mute fayllar qo'shish
    mute_base = f"{now_dir}/logs/mute"
    for _ in range(2):
        opt.append(
            f"{mute_base}/0_gt_wavs/mute{int(sample_rate.rstrip('k'))*1000}.wav|"
            f"{mute_base}/3_feature768/mute.npy|"
            f"{mute_base}/2a_f0/mute.wav.npy|"
            f"{mute_base}/2b-f0nsf/mute.wav.npy|0"
        )
    shuffle(opt)
    with open(f"{exp_dir}/filelist.txt", "w") as f:
        f.write("\n".join(opt))
    return True, f"{len(opt)-2} ta fragment"

def find_outputs(model_name):
    pth = (
        glob.glob(f"{now_dir}/assets/weights/{model_name}*.pth") +
        glob.glob(f"{now_dir}/logs/{model_name}/*.pth")
    )
    idx = (
        glob.glob(f"{now_dir}/logs/{model_name}/*.index") +
        glob.glob(f"{now_dir}/assets/indices/{model_name}*.index")
    )
    return list(set(pth)), list(set(idx))

def build_zip(model_name, pth_files, idx_files):
    out = f"/content/{model_name}_ElbekStudiya.zip"
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in pth_files + idx_files:
            if os.path.exists(f):
                z.write(f, os.path.basename(f))
    return out if os.path.exists(out) and os.path.getsize(out) > 100 else None

# ═══════════════════════════════════════════════════════════════════════
#  ASOSIY O'QITISH FUNKSIYASI
# ═══════════════════════════════════════════════════════════════════════

def train_model(
    raw_name,
    audio_files,
    total_epoch,
    save_epoch,
    sample_rate,
    f0_method,
    pretrain_choice,
    batch_size,
    fp16_run,
    progress=gr.Progress(track_tqdm=True)
):
    log = []
    def L(msg):
        log.append(str(msg))
        return '\n'.join(log)

    if not raw_name or not raw_name.strip():
        return '❌ Model nomini kiriting!', None
    if not audio_files:
        return '❌ Ovoz faylini yuklang!', None

    model_name = clean_name(raw_name)

    L('═' * 55)
    L('🎙️  Elbek Studiya — RVC Trainer')
    L('═' * 55)
    L(f'🏷️  Model     : {model_name}')
    L(f'🔢  Epoch     : {total_epoch}')
    L(f'📻  SR        : {sample_rate}')
    L(f'🎵  F0 method : {f0_method}')
    L(f'🧠  Pretrain  : {pretrain_choice}')
    L(f'📦  Batch     : {batch_size}')
    L(f'📁  Fayllar   : {len(audio_files)} ta')
    L('─' * 55)

    try:
        os.chdir(ROOT_DIR)
        sr_num = int(sample_rate.rstrip('k')) * 1000
        exp_dir = f"{now_dir}/logs/{model_name}"
        os.makedirs(exp_dir, exist_ok=True)

        # ── 1. Dataset tayyorlash ──────────────────────────────────────
        progress(0.03, desc='📁 Dataset tayyorlanmoqda...')
        ds = '/content/dataset'
        # Avvalgi fayllarni o'chirish
        for f in os.listdir(ds):
            os.remove(os.path.join(ds, f))
        for f in audio_files:
            shutil.copy(f.name, os.path.join(ds, os.path.basename(f.name)))
        L(f'\n✅ [1/5] Dataset: {len(audio_files)} fayl → {ds}')

        # ── 2. Preprocessing ──────────────────────────────────────────
        progress(0.10, desc='🔊 Preprocessing...')
        L('\n⏳ [2/5] Preprocessing (slice + resample)...')
        preprocess_script = f"{ROOT_DIR}/infer/modules/train/preprocess.py"
        percentage = 3.0
        with open(f"{exp_dir}/logfile.log", "a") as logf:
            r = subprocess.run(
                f"python {preprocess_script} {ds} {sr_num} 2 {exp_dir} False {percentage}",
                shell=True, cwd=ROOT_DIR, text=True,
                stdout=logf, stderr=logf
            )
        with open(f"{exp_dir}/logfile.log", "r") as logf:
            log_content = logf.read()
        if "успешно" in log_content or "success" in log_content.lower():
            L('✅ Preprocessing tayyor')
        else:
            L(f'⚠️  Preprocessing (rc={r.returncode}). Davom etamiz...')
            L(log_content[-800:])

        # ── 3. F0 extraction ──────────────────────────────────────────
        progress(0.25, desc=f'🎵 F0 ({f0_method})...')
        L(f'\n⏳ [3/5] F0 chiqarilmoqda ({f0_method})...')
        f0_script = f"{ROOT_DIR}/infer/modules/train/extract_f0.py"
        with open(f"{exp_dir}/logfile.log", "a") as logf:
            r = subprocess.run(
                f"python {f0_script} 1 0 0 {exp_dir} False {f0_method}",
                shell=True, cwd=ROOT_DIR, text=True,
                stdout=logf, stderr=logf
            )
        with open(f"{exp_dir}/logfile.log", "r") as logf:
            log_content = logf.read()
        if "извлечен" in log_content or "done" in log_content.lower() or r.returncode == 0:
            L('✅ F0 chiqarildi')
        else:
            L(f'⚠️  F0 (rc={r.returncode})')

        # ── 4. Feature extraction ─────────────────────────────────────
        progress(0.42, desc='🧠 Feature extraction...')
        L('\n⏳ [4/5] Feature extraction (HuBERT)...')
        feat_script = f"{ROOT_DIR}/infer/modules/train/extract_feature.py"
        with open(f"{exp_dir}/logfile.log", "a") as logf:
            r = subprocess.run(
                f"python {feat_script} 1 0 {exp_dir} v2 False",
                shell=True, cwd=ROOT_DIR, text=True,
                stdout=logf, stderr=logf
            )
        with open(f"{exp_dir}/logfile.log", "r") as logf:
            log_content = logf.read()
        if "извлечены" in log_content or r.returncode == 0:
            L('✅ Feature extraction tayyor')
        else:
            L(f'⚠️  Feature extraction (rc={r.returncode})')

        # ── 4b. Index extraction ──────────────────────────────────────
        progress(0.50, desc='📑 Index...')
        L('\n⏳ [4b] Index yaratilmoqda...')
        idx_script = f"{ROOT_DIR}/infer/modules/train/extract_index.py"
        r = subprocess.run(
            f"python {idx_script} {exp_dir} v2 Auto",
            shell=True, cwd=ROOT_DIR, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        L(r.stdout[-1000:] if r.stdout else '(natija yo\'q)')

        # ── 5. Pretrain yuklab olish ──────────────────────────────────
        progress(0.55, desc='📥 Pretrain yuklanmoqda...')
        L(f'\n⏳ [5a/5] Pretrain yuklanmoqda ({pretrain_choice})...')
        pretrain_outpath = "/content/pretrained_models"
        G_file, D_file = download_pretrain(sample_rate, pretrain_choice, pretrain_outpath)
        if os.path.exists(G_file):
            L(f'✅ Pretrain: {os.path.basename(G_file)}')
        else:
            L(f'⚠️  Pretrain yuklanmadi, standart ishlatiladi')

        # ── 5b. Filelist yaratish ─────────────────────────────────────
        progress(0.60, desc='📝 Filelist...')
        L('\n⏳ [5b/5] Filelist yaratilmoqda...')
        ok, msg = build_filelist(model_name, sample_rate, exp_dir)
        if ok:
            L(f'✅ Filelist: {msg}')
        else:
            L(f'⚠️  Filelist: {msg}')

        # Config fayl
        config_path = f"configs/v2/{sample_rate}.json"
        config_save = f"{exp_dir}/config.json"
        if not os.path.exists(config_save) and os.path.exists(config_path):
            with open(config_path, 'r') as cf:
                cfg = json.load(cf)
            cfg['train']['fp16_run'] = fp16_run
            cfg['train']['log_interval'] = 1
            with open(config_save, 'w', encoding='utf-8') as cf:
                json.dump(cfg, cf, ensure_ascii=False, indent=4)

        # ── 5c. Model o'qitish ────────────────────────────────────────
        progress(0.65, desc=f'🏋️ O\'qitish ({total_epoch} epoch)...')
        L(f'\n⏳ [5c/5] Model o\'qitilmoqda ({total_epoch} epoch, batch={batch_size})...')
        L('⏳ Bu biroz vaqt oladi, sabr qiling...')

        train_script = f"{ROOT_DIR}/infer/modules/train/train.py"
        G_arg = f"-pg {G_file}" if os.path.exists(G_file) else ""
        D_arg = f"-pd {D_file}" if os.path.exists(D_file) else ""
        cmd = (
            f"python {train_script} "
            f"-e '{model_name}' "
            f"-sr {sample_rate} "
            f"-bs {batch_size} "
            f"-te {total_epoch} "
            f"-se {save_epoch} "
            f"{G_arg} {D_arg} "
            f"-o AdamW"
        )
        p = Popen(cmd, shell=True, cwd=ROOT_DIR, stdout=PIPE, stderr=STDOUT,
                  bufsize=1, universal_newlines=True)
        output_lines = []
        for line in p.stdout:
            stripped = line.strip()
            output_lines.append(stripped)
            # Faqat muhim qatorlarni log ga qo'shish
            if any(k in stripped for k in ['loss', 'epoch', 'step', 'saved', 'Error', 'Traceback']):
                log.append(stripped)
        p.wait()

        last_lines = output_lines[-20:]
        L('\n'.join(last_lines))

        if p.returncode == 0:
            L('✅ O\'qitish yakunlandi!')
        else:
            L(f'⚠️  O\'qitish xato (rc={p.returncode})')

        # ── 6. Fayllarni topish va ZIP ────────────────────────────────
        progress(0.95, desc='📦 ZIP yaratilmoqda...')
        pth_f, idx_f = find_outputs(model_name)

        L('\n🔍 Fayllar qidirilmoqda...')
        L(f'   .pth   : {[os.path.basename(f) for f in pth_f] or "topilmadi"}')
        L(f'   .index : {[os.path.basename(f) for f in idx_f] or "topilmadi"}')

        if not pth_f and not idx_f:
            # Logs papkasini ko'rsatish
            all_f = glob.glob(f'{exp_dir}/**', recursive=True)
            L(f'\n📂 Logs ({exp_dir}):')
            for ff in sorted(all_f)[:25]:
                L(f'   {ff}')
            return '\n'.join(log), None

        zip_path = build_zip(model_name, pth_f, idx_f)
        if not zip_path:
            L('❌ ZIP yaratishda xato!')
            return '\n'.join(log), None

        L('\n' + '█'*55)
        L('🎉  O\'QITISH MUVAFFAQIYATLI YAKUNLANDI!')
        L('█'*55)
        L(f'📦  ZIP: {zip_path}')
        L(f'   .pth   → {[os.path.basename(f) for f in pth_f]}')
        L(f'   .index → {[os.path.basename(f) for f in idx_f]}')
        L('⬇️   Quyidagi yuklab olish tugmasini bosing!')

        progress(1.0, desc='🎉 Tayyor!')
        return '\n'.join(log), zip_path

    except Exception:
        import traceback
        L(f'\n💥 Xato:\n{traceback.format_exc()}')
        return '\n'.join(log), None

# ═══════════════════════════════════════════════════════════════════════
#  GRADIO INTERFEYS
# ═══════════════════════════════════════════════════════════════════════

CSS = """
body, .gradio-container { background:#0a0a12 !important; font-family:'Segoe UI',system-ui,sans-serif !important; }
.es-top {
    background:linear-gradient(135deg,#12001f 0%,#1a0030 50%,#0d0025 100%);
    border:2px solid #c0392b; border-radius:18px; padding:32px 24px 24px;
    text-align:center; margin-bottom:18px;
    box-shadow:0 0 40px rgba(192,57,43,.4),inset 0 1px 0 rgba(255,255,255,.05);
}
.es-top h1 { color:#fff; font-size:2.4em; font-weight:800; margin:0 0 6px;
    text-shadow:0 0 20px rgba(192,57,43,.8); letter-spacing:1px; }
.es-top .r { color:#e74c3c; }
.es-top p  { color:#9b59b6; font-size:1em; margin:0; }
.es-top .chip { display:inline-block; background:#c0392b; color:#fff;
    border-radius:20px; padding:4px 14px; font-size:.78em; font-weight:700; margin-top:10px; }
.gr-button.gr-button-primary {
    background:linear-gradient(135deg,#c0392b,#922b21) !important;
    border:none !important; font-size:1.15em !important; padding:15px !important;
    box-shadow:0 4px 20px rgba(192,57,43,.5) !important; border-radius:10px !important;
    transition:all .25s !important; font-weight:800 !important;
}
.gr-button.gr-button-primary:hover { transform:translateY(-2px) scale(1.01) !important;
    box-shadow:0 8px 30px rgba(192,57,43,.7) !important; }
label,.label-wrap span { color:#dfe6e9 !important; font-weight:600 !important; font-size:.92em !important; }
textarea,input[type=text] { background:#1a1a2e !important; color:#ecf0f1 !important; border-color:#34495e !important; }
.footer { text-align:center; color:#636e72; font-size:.8em; padding:12px; border-top:1px solid #2d3436; margin-top:14px; }
"""

TOP_HTML = """
<div class='es-top'>
  <h1>🎙️ <span class='r'>Elbek Studiya</span> RVC Model Trainer</h1>
  <p>Ovoz yuklang · Model nomi kiriting · Train bosing · ZIP yuklab oling</p>
  <span class='chip'>⚡ Retrieval-based Voice Conversion v2</span>
</div>
"""

theme = gr.themes.Base(
    primary_hue=gr.themes.colors.red,
    secondary_hue=gr.themes.colors.purple,
    neutral_hue=gr.themes.colors.gray,
).set(
    body_background_fill='#0a0a12',
    block_background_fill='#13131f',
    block_border_color='#2d2d44',
    input_background_fill='#1a1a2e',
    button_primary_background_fill='#c0392b',
    button_primary_background_fill_hover='#e74c3c',
    button_primary_text_color='white',
)

with gr.Blocks(css=CSS, theme=theme, title='🎙️ Elbek Studiya RVC') as demo:

    gr.HTML(TOP_HTML)

    with gr.Row(equal_height=False):

        # ━━━━ CHAP: KIRISH ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        with gr.Column(scale=5, min_width=340):
            gr.Markdown('### 📋 Model sozlamalari')

            model_inp = gr.Textbox(
                label='🏷️ Model nomi',
                placeholder='masalan:  sarvar_ovozi   yoki   elbek_2024',
                max_lines=1,
                info='Faqat harflar, raqamlar va _ belgisi'
            )
            audio_inp = gr.File(
                label='🎤 Ovoz fayllari  (.wav / .mp3 / .flac)',
                file_count='multiple',
                file_types=['.wav','.mp3','.flac','.ogg','.m4a'],
            )

            with gr.Accordion('⚙️ Kengaytirilgan sozlamalar', open=True):
                with gr.Row():
                    epoch_inp = gr.Slider(
                        label='🔢 Epoch soni',
                        minimum=10, maximum=1000, value=300, step=10,
                        info='Tavsiya: 300-1000'
                    )
                    save_inp = gr.Slider(
                        label='💾 Har necha epochda saqlash',
                        minimum=10, maximum=200, value=310, step=10,
                        info='Oxirida bir marta saqlash uchun epoch+10'
                    )
                with gr.Row():
                    sr_inp = gr.Radio(
                        label='📻 Sample Rate',
                        choices=['40k','48k'],
                        value='40k'
                    )
                    f0_inp = gr.Dropdown(
                        label='🎵 F0 (pitch) metodi',
                        choices=['rmvpe','rmvpe+','harvest'],
                        value='rmvpe',
                        info='rmvpe = tavsiya etiladi'
                    )
                with gr.Row():
                    pretrain_inp = gr.Dropdown(
                        label='🧠 Pretrained model',
                        choices=['Default','Snowie v3.1'],
                        value='Default',
                        info='Default = standart'
                    )
                    batch_inp = gr.Slider(
                        label='📦 Batch size',
                        minimum=2, maximum=26, value=8, step=2,
                        info='T4 GPU: 8-12'
                    )
                fp16_inp = gr.Checkbox(
                    label='⚡ FP16 (xotira tejash)',
                    value=True,
                    info='Yoqilgan = tezroq, biroz past sifat'
                )

            train_btn = gr.Button(
                '🚀   O\'QITISHNI BOSHLASH',
                variant='primary', size='lg'
            )

            gr.Markdown("""
---
**📌 Qo'llanma:**  
① Model nomi kiriting  
② `.wav`/`.mp3` ovoz yuklang (10+ daqiqa tavsiya)  
③ Epoch tanlang (300–1000)  
④ **Train** bosing → tayyor bo'lgach ZIP yuklab oling  

**💡 Yaxshi natija uchun:** 10-20 daqiqa sifatli, shovqinsiz ovoz
""")

        # ━━━━ O'NG: NATIJA ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        with gr.Column(scale=5, min_width=340):
            gr.Markdown('### 📊 Jarayon va natija')

            log_out = gr.Textbox(
                label='📜 Jarayon jurnali',
                lines=28,
                interactive=False,
                show_copy_button=True,
                placeholder=(
                    'O\'qitish boshlanganidan keyin bu yerda ko\'rinadi:\n\n'
                    '  ══════════════════════════════\n'
                    '  🎙️  Elbek Studiya — RVC Trainer\n'
                    '  ══════════════════════════════\n'
                    '  ✅ [1/5] Dataset tayyor\n'
                    '  ✅ [2/5] Preprocessing\n'
                    '  ✅ [3/5] F0 chiqarildi\n'
                    '  ✅ [4/5] Feature extraction\n'
                    '  ✅ [5/5] Model o\'qitildi\n'
                    '  ████████████████████████████\n'
                    '  🎉  MUVAFFAQIYATLI YAKUNLANDI!'
                )
            )

            zip_out = gr.File(
                label='📦 Tayyor model — ZIP yuklab olish (.pth + .index)',
                interactive=False
            )

            gr.Markdown("""
---
**📦 ZIP tarkibi:**
```
model_nomi_ElbekStudiya.zip
├── model_nomi.pth           ← asosiy model
└── added_model_nomi_v2.index ← qidiruv indeksi
```
**🔗 Foydalanish:**  
[RVC Simple (HuggingFace)](https://huggingface.co/spaces/r3gm/RVC_HF)
""")

    gr.HTML("""
<div class='footer'>
  🎙️ <strong>Elbek Studiya RVC Trainer</strong> &nbsp;•&nbsp;
  Powered by <a href='https://github.com/Bebra777228/TrainVocModel-EN'
    style='color:#e74c3c;text-decoration:none'>Bebra777228/TrainVocModel-EN</a> &nbsp;•&nbsp;
  UI: <a href='https://gradio.app' style='color:#9b59b6;text-decoration:none'>Gradio</a>
</div>
""")

    train_btn.click(
        fn=train_model,
        inputs=[model_inp, audio_inp, epoch_inp, save_inp,
                sr_inp, f0_inp, pretrain_inp, batch_inp, fp16_inp],
        outputs=[log_out, zip_out],
        show_progress=True
    )

print('\n\033[1;32m🚀 Gradio UI ishga tushmoqda...\033[0m')
print('\033[1;33m⏳ Bir daqiqa kuting...\033[0m\n')

demo.queue(max_size=3).launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=True,
    debug=False,
    show_error=True,
    quiet=False,
    inbrowser=False
)